In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.summary


>DOM summary 
output-file: core.dom.summary
title: core.dom.summary

This notebook provides utility functions for DOM summary
---

In [ ]:
# | default_exp core.dom.summary

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import os
import re
from collections.abc import Callable
from enum import Enum
from functools import wraps
from inspect import Parameter, signature
from pathlib import Path
from typing import Optional

from pydantic import BaseModel, Field

from ollama import AsyncClient
from openai import AsyncOpenAI

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
    ConfigNode,
)

from ribosome.core.dom.utils import get_summary_response_async, get_image_summary_response_async, image_link_pattern,get_image_summary_async

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
#| export
ResponseStatus = Enum("ResponseStatus", ["PENDING", "PROCESSING", "REORGNIZED", "SEMANTICIZED", "CANCELLED", "COMPLETED", "ERROR"])
class AnalysisStatus(BaseModel):
    """
    Enum to represent the status of the analysis process.
    """
    status: ResponseStatus = Field(ResponseStatus.PENDING, description="Current status of the analysis process")
    exception: str = Field("", description="Exception message if any error occurs during analysis")

def _config_error(message: str, config_node: ConfigNode, node=None) -> ValueError:
    line = jsoncfg.node_location(config_node).line if config_node is not None else "unknown"
    details = [message, f"On line: {line}"]
    if node is not None:
        details.insert(1, f"Node: {node}")
    return ValueError("\n".join(details))

def with_async_context(context_builder: Callable[..., str]) -> Callable:
    """Wrap async helpers to append contextual information to unexpected errors."""
    def build_context(*args, **kwargs) -> str:
        try:
            return context_builder(*args, **kwargs)
        except TypeError:
            params = tuple(signature(context_builder).parameters.values())
            if any(param.kind is Parameter.VAR_POSITIONAL for param in params):
                context_args = args
            else:
                positional_limit = sum(
                    param.kind in (Parameter.POSITIONAL_ONLY, Parameter.POSITIONAL_OR_KEYWORD)
                    for param in params
                )
                context_args = args[:positional_limit]

            if any(param.kind is Parameter.VAR_KEYWORD for param in params):
                context_kwargs = kwargs
            else:
                allowed_kwargs = {
                    param.name
                    for param in params
                    if param.kind in (Parameter.KEYWORD_ONLY, Parameter.POSITIONAL_OR_KEYWORD)
                }
                context_kwargs = {key: value for key, value in kwargs.items() if key in allowed_kwargs}

            return context_builder(*context_args, **context_kwargs)

    def decorator(func: Callable) -> Callable:
        @wraps(func)
        async def wrapper(*args, **kwargs):
            try:
                return await func(*args, **kwargs)
            except ValueError:
                raise
            except Exception as exc:
                raise ValueError(f"{build_context(*args, **kwargs)}\nError: {exc}") from exc
        return wrapper
    return decorator

In [ ]:
# | export
def _map_tree(node, transform: Callable):
    """
    Recursively apply a transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.

    Returns:
        The transformed tree.
    """
    
    node = transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    _map_tree(child, transform)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = _map_tree(value, transform)
    elif isinstance(node, list):
        node = [
            _map_tree(child, transform) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node

In [ ]:
# | export
async def _map_tree_async(node, transform: Callable, on_exit: Optional[Callable] = None):
    """Recursively apply an asynchronous transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.
        on_exit: An optional callable that is called with the node after it has been transformed.

    Returns:
        The transformed tree.
    """
    
    node = await transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    await _map_tree_async(child, transform, on_exit)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = await _map_tree_async(value, transform, on_exit)
    elif isinstance(node, list):
        node = [
            await _map_tree_async(child, transform, on_exit) if isinstance(child, (dict, list)) else child
            for child in node
        ]

    if on_exit is not None:
        await on_exit(node)
    return node


In [ ]:
# | export
async def get_leaf_summary_async(node:str, min_len:int, client: AsyncClient | AsyncOpenAI, model:str) -> str: # If the string length is less than 200, return the node as is
    """ Generate a summary for a given text node.

    Args:
        node (str): The text node to summarize.
        min_len (int): The minimum length of the text node to summarize.

    Returns:
        str: The summary of the text node.
    """
    
    if len(node) < min_len:
        return node
    # If the node is not an image link, return a summary of the text using unified function
    response_content = await get_summary_response_async(client, node, model=model, role="user", lang='zh')
    if response_content:
        return response_content
    else:
        # If the response is empty, return the original node
        return node


In [ ]:
# | export
async def get_list_summary_async(root: list, leaf_min_len: int, client: AsyncClient | AsyncOpenAI, model:str) -> str:
    """
    Given a list of strings, return a summary of the list.
    If the list is empty, return an empty string.
    If the list has only one element, return the summary of that element.
    If the list has more than one element, return a summary of the concatenated elements.
    """
    list_summary = []
    for n in root:
        if isinstance(n, str):
            # If the element is a string, get its summary
            summary = await get_leaf_summary_async(n, min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, int) or isinstance(n, float):
            # If the element is a number, get its summary
            summary = await get_leaf_summary_async(str(n), min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, dict):
            # If the dict has a 's' key, use it as the summary
            summary = n.get('s', '')
        elif isinstance(n, list):
            # If the element is a list, get its summary
            summary = await get_list_summary_async(n, leaf_min_len=leaf_min_len, client=client, model=model)
        elif n is None:
            # If the element is None, skip it
            summary = ""
        else:
            raise ValueError(f"Unsupported element type: {type(n)} in {n}")
        list_summary.append(summary)
        
    # Concatenate all elements and summarize
    # concatenated = " ".join(list_summary)
    # response = get_text_summary_response(concatenated, model="gemma3:27b", role="user", lang='zh')
    # if response.message.content:
    #     return response.message.content
    # else:
    #     # If the response is empty, raise an exception
    #     raise ValueError("Summary response is empty. Please check the input data.")
    return await get_leaf_summary_async(" ".join(list_summary), min_len=leaf_min_len, client=client, model=model)


In [ ]:
# | export
def _resolve_image_link(node: dict, config_node: ConfigNode, root_path: Path) -> str:
    """Resolve the link of an image node.

    Args:
        self (DOMClass): The DOMClass instance.
        node (dict): The image node.
        config_node (ConfigNode): The configuration node associated with the image.

    Returns:
        str: The resolved image link.
    """
    try:
        image_link = root_path / node['c'][2][0]
    except (IndexError, KeyError, TypeError) as exc:
        raise _config_error("Invalid image node structure", config_node, node) from exc

    image_link = str(image_link)
    if not image_link or not re.search(image_link_pattern, image_link):
        raise _config_error(f"Invalid image link: {image_link}", config_node, node)
    return image_link



In [ ]:
# | export
def _config_node_line(config_node: ConfigNode) -> int | str:
    """Get the line number of a ConfigNode."""
    
    try:
        return jsoncfg.node_location(config_node).line
    except Exception:
        return "unknown"

# def _config_error(message: str, config_node: ConfigNode, node=None) -> ValueError:
#     """Create a ValueError with detailed information about a ConfigNode."""
#     details = [
#         message,
#         f"ConfigNode: {config_node} of type {type(config_node)}.",
#         f"On line: {_config_node_line(config_node)}",
#     ]
#     if node is not None:
#         details.insert(1, f"Node: {node}")
#     return ValueError("\n".join(details))


In [ ]:
# | export
@with_async_context(lambda config_node, node: str(_config_error("Error summarizing image node", config_node, node)))
async def _summarize_image_node_async(config_node: ConfigNode,node: dict,root_path: Path, client: AsyncClient|AsyncOpenAI, model: str) -> dict:
    """Summarize an image node asynchronously.

    Args:
        self (DOMClass): The DOMClass instance.
        node (dict): The image node.
        config_node (ConfigNode): The configuration node associated with the image.

    Returns:
        dict: The summarized image node.
    """
    if not isinstance(node.get('c', []), list):
        raise _config_error("Invalid image caption format", config_node, node)

    image_link = _resolve_image_link(node, config_node,root_path)
    node["s"] = await get_image_summary_async(
        client=client,
        image_link=image_link,
        model=model,
        role="user",
        lang='zh'
    )
    print(f"Summarize image: {image_link} on line {_config_node_line(config_node)}")
    return node


In [ ]:
# | export
async def _summarize_leaf_async(node: str, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Summarize a leaf node asynchronously.

    Args:
        leaf_min_len: The minimum length of the leaf node text.
        node: The leaf node to summarize.
        min_len: The minimum length of the text to summarize.

    Returns:
        The summarized text.
    """
    text = str(node).strip()
    threshold = leaf_min_len if min_len is None else min_len
    if len(text) < threshold:
        return text

    response_content = await get_summary_response_async(
        client,
        text,
        model=model,
        role="user",
        lang=lang,
    )
    return response_content or text


In [ ]:
# | export
async def _coerce_summary_async(value: str|int|float|dict|list|None, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Coerce a value into a summarized string.

    Args:
        self (DOMClass): The DOMClass instance.
        value: The value to be coerced into a summarized string.

    Raises:
        ValueError: If the value type is unsupported.

    Returns:
        str: The summarized string.
    """
    if isinstance(value, str):
        return await _summarize_leaf_async(value, leaf_min_len=0, min_len=None, client=client, model=model, lang=lang)
    if isinstance(value, (int, float)):
        return await _summarize_leaf_async(str(value), leaf_min_len=0, min_len=None, client=client, model=model, lang=lang)
    if isinstance(value, dict):
        return value.get('s', '')
    if isinstance(value, list):
        return await _summarize_list_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if value is None:
        return ""
    raise ValueError(f"Unsupported element type: {type(value)} in {value}")


In [ ]:
# | export
async def _summarize_list_async(root: list, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Summarize a list of values asynchronously.

    Args:
        self (DOMClass): The DOMClass instance.
        root (list): The list of values to be summarized.

    Returns:
        str: The summarized string.
    """
    parts = [await _coerce_summary_async(item, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang) for item in root]
    merged = " ".join(part for part in parts if part)
    return await _summarize_leaf_async(merged, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)


In [ ]:
# | export
def _record_summary_progress(node_type: str, config_node: ConfigNode, file_path: Path, table_count: int, section_count: int) -> None:
    """Record the progress of summarizing a node.

    Args:
        node_type (str): The type of the node being summarized.
        config_node (ConfigNode): The configuration node associated with the summary.
    """
    if node_type == "Table":
        print(f"{file_path} Summarize table: {table_count} on line {_config_node_line(config_node)}")
        table_count += 1
    elif node_type == "Section":
        print(f"{file_path} Summarize section: {section_count} on line {_config_node_line(config_node)}")
        section_count += 1


In [ ]:
# | export
@with_async_context(lambda config_node, node: str(_config_error("Error summarizing node", config_node, node)))
async def summary_node_pair_async(
    config_node: ConfigNode,
    node: dict | list | str | int | float | bool | None,
    root_path: Path,
    client: AsyncClient|AsyncOpenAI,
    model: str,
    file_path: Path,
    table_count: int,
    section_count: int,
    action: Optional[Callable] = None,
    leaf_min_len: int = 0,
    min_len: Optional[int] = None,
    lang: str='zh',
) -> tuple[ConfigNode, dict | list | str | int | float | bool | None]:
    """ Generate a summary for a given node.

    Args:
        config_node (ConfigNode): The configuration node associated with the node to summarize.
        node (dict | list | str | int | float | bool | None): The node to summarize.
        action (Optional[Callable], optional): An optional action to apply to the node before summarizing. Defaults to None.

    Raises:
        self._config_error: Raises an error if the node cannot be summarized.
        
    Returns:
        tuple[ConfigNode, dict | list | str | int | float | bool | None]: A tuple containing the configuration node and the summarized node.
    """

    if action:
        node = action(node)

    if isinstance(node, dict):
        if not isinstance(config_node, ConfigJSONObject):
            raise _config_error(f"Expected ConfigJSONObject, got {type(config_node)}", config_node, node)

        t = node.get("t")
        if t is None:
            raise _config_error("Node does not have a 't' key", config_node, node)

        if t == "Image":
            node = await _summarize_image_node_async(config_node, node, root_path=root_path, client=client,model=model)
        elif t in {"Cite", "AlignDefault", "ColWidth"}:
            node["s"] = ""
        else:
            content = node.get("c")
            if not content:
                node["s"] = ""
            else:
                dict_summary = []
                for (_, cvalue), (key, value) in zip(config_node, node.items()):
                    if key == "t":
                        continue
                    if isinstance(cvalue, ConfigJSONArray):
                        if not isinstance(value, list):
                            raise _config_error(f"Expected a list for key {key}, got {type(value)}", cvalue, value)
                        if not value:
                            continue
                        children = []
                        for citem, item in zip(cvalue, value):
                            _, child = await summary_node_pair_async(citem, item, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
                            children.append(child)
                        node[key] = children
                        dict_summary.append(await _summarize_list_async(children, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang))
                    elif isinstance(cvalue, ConfigJSONObject):
                        if not isinstance(value, dict):
                            raise _config_error(f"Expected a dict for key {key}, got {type(value)}", cvalue, value)
                        _, child = await summary_node_pair_async(cvalue, value, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
                        node[key] = child
                        if isinstance(child, dict) and 's' in child:
                            dict_summary.append(child['s'])
                    elif value is None or value == "":
                        continue
                    elif isinstance(cvalue, ConfigJSONScalar):
                        if not isinstance(value, (str, int, float, bool)):
                            raise _config_error(f"Expected a scalar for key {key}, got {type(value)}", cvalue, value)
                        dict_summary.append(await _summarize_leaf_async(str(value), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang))

                joined_summary = " ".join(part for part in dict_summary if part)
                node["s"] = await _summarize_leaf_async(joined_summary, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang) if joined_summary else ""

            _record_summary_progress(t, config_node, file_path=file_path, table_count=table_count, section_count=section_count)

    elif isinstance(node, list):
        if not isinstance(config_node, ConfigJSONArray):
            raise _config_error(f"Expected ConfigJSONArray, got {type(config_node)}", config_node, node)
        new_node = []
        for citem, item in zip(config_node, node):
            if isinstance(item, (dict, list)):
                _, child = await summary_node_pair_async(citem, item, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
            elif isinstance(item, (str, int, float, bool)):
                child = str(item).strip()
            else:
                child = item
            new_node.append(child)
        node = new_node

    elif isinstance(node, (str, int, float, bool)):
        if not isinstance(config_node, ConfigJSONScalar):
            raise _config_error(f"Expected ConfigJSONScalar, got {type(config_node)}", config_node, node)
        node = await _summarize_leaf_async(str(node), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    elif node is not None:
        raise _config_error(f"Unsupported node type: {type(node)}", config_node, node)

    return config_node, node

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

## Summary Tests

Fastcore-style checks for the exported summary helpers and async traversal functions.

In [ ]:
# | hide
import json
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace
from typing import Optional
from unittest.mock import patch

import jsoncfg
from fastcore.test import test_eq

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import ribosome.core.dom.summary as summary_mod

summary_mod.json = json
summary_mod.Optional = Optional


class FakeSummaryContext:
    leaf_min_len = 3

    def __init__(self, root: Path):
        self.root_path = root
        self.modelroot_path = root
        self.file_path = root / "doc.md"
        self.file_path.write_text("# doc\n", encoding="utf-8")
        self.ast_json_file = root / "doc_ast.json"
        self.ast_json = None
        self.title = "doc"
        self.table_count = 0
        self.section_count = 0
        self.openai_client = SimpleNamespace(name="fake-openai")
        self.ollama_client = SimpleNamespace(name="fake-ollama")

    @classmethod
    def identity(cls, obj):
        return obj

    def _config_error(self, message, config_node, node=None):
        line = jsoncfg.node_location(config_node).line if config_node is not None else "unknown"
        details = [message, f"On line: {line}"]
        if node is not None:
            details.insert(1, f"Node: {node}")
        return ValueError("\n".join(details))

    async def _summarize_leaf_async(self, text: str):
        return await summary_mod.get_leaf_summary_async(text, self.leaf_min_len)

    async def _summarize_list_async(self, root: list):
        return await summary_mod.get_list_summary_async(root)

    async def _summarize_image_node_async(self, node: dict, config_node):
        updated = dict(node)
        updated["s"] = f"IMG<{Path(node['c'][2][0]).name}>"
        return updated

    def _record_summary_progress(self, node_type: str, config_node):
        if node_type == "Table":
            self.table_count += 1
        elif node_type == "Section":
            self.section_count += 1


def make_config(payload: dict | list) -> tuple[tempfile.TemporaryDirectory, object]:
    tmpdir = tempfile.TemporaryDirectory()
    path = Path(tmpdir.name) / "config.json"
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    return tmpdir, jsoncfg.load_config(str(path))


async def fake_text_summary(client, content, model="unused", role="user", lang="zh"):
    return f"TXT<{content}>"


async def fake_image_summary(client, image_link, model="unused", role="user", lang="zh"):
    return f"IMG<{Path(image_link).name}>"


async def fake_text_response(client, content, model="unused", role="user", lang="zh"):
    return SimpleNamespace(message=SimpleNamespace(content=f"TXTRESP<{content}>"))


async def fake_image_response(client, image_link, model="unused", role="user", lang="zh"):
    return SimpleNamespace(message=SimpleNamespace(content=f"IMGRESP<{Path(image_link).name}>"))

In [ ]:
# | hide
status = AnalysisStatus()
test_eq(status.status, summary_mod.ResponseStatus.PENDING)
test_eq(status.exception, "")

tmpdir = tempfile.TemporaryDirectory()
ctx = FakeSummaryContext(Path(tmpdir.name))
summary_mod.self = ctx
summary_mod.action = lambda node: node
summary_mod.image_link_pattern = r"[^\s]+\.(?:jpg|jpeg|png|gif|bmp|webp)"

async def passthrough(value):
    return value

decorated = summary_mod.with_async_context(lambda value: f"ctx={value}")(passthrough)
test_eq(await decorated("ok"), "ok")

@summary_mod.with_async_context(lambda value: f"ctx={value}")
async def wrapped_failure(value):
    raise RuntimeError("boom")

wrapped_error = None
try:
    await wrapped_failure("x")
except ValueError as exc:
    wrapped_error = exc
test_eq("ctx=x" in str(wrapped_error), True)
test_eq("boom" in str(wrapped_error), True)

with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True):
    test_eq(await summary_mod.get_leaf_summary_async("hi", 5), "hi")
    test_eq(await summary_mod.get_leaf_summary_async("hello", 3), "TXT<hello>")
    test_eq(await summary_mod.get_list_summary_async(["alpha", 2, {"s": "beta"}, None]), "TXT<TXT<alpha> 2 beta >")

list_error = None
try:
    await summary_mod.get_list_summary_async([object()])
except ValueError as exc:
    list_error = exc
test_eq("Unsupported element type" in str(list_error), True)

config_tmpdir, config_scalar = make_config({"value": "text"})
with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True):
    scalar_config, scalar_node = await summary_mod.summary_node_pair_async(config_scalar["value"], "text")
test_eq(str(scalar_node), "TXT<text>")
config_tmpdir.cleanup()

config_tmpdir, config_list = make_config({"items": ["text", 2, {"t": "Cite", "c": []}]})
with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True):
    _, list_node = await summary_mod.summary_node_pair_async(config_list["items"], ["text", 2, {"t": "Cite", "c": []}], action=lambda node: node)
test_eq(list_node[0], "text")
test_eq(list_node[1], "2")
test_eq(list_node[2]["s"], "")
config_tmpdir.cleanup()

section_payload = {
    "t": "Section",
    "c": [
        {"t": "Header", "c": [1, ["intro", [], []], [{"t": "Str", "c": "Intro"}]]},
        {"t": "Content", "c": [{"t": "Para", "c": [{"t": "Str", "c": "Body"}]}]},
    ],
}
config_tmpdir, config_section = make_config(section_payload)
with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True):
    _, summarized_section = await summary_mod.summary_node_pair_async(config_section, section_payload)
test_eq(summarized_section["t"], "Section")
test_eq("s" in summarized_section, True)
test_eq(ctx.section_count, 1)
config_tmpdir.cleanup()

ctx.table_count = 0
ctx.section_count = 0
ctx.ast_json = json.dumps({"blocks": [section_payload]}, ensure_ascii=False)
ctx.ast_json_file.write_text(ctx.ast_json, encoding="utf-8")
with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True):
    await summary_mod.summary_nodewline_main()
summary_doc = json.loads(ctx.ast_json)
test_eq(summary_doc["title"], "doc")
test_eq(summary_doc["file_path"], str(ctx.file_path))
test_eq("summary" in summary_doc, True)

with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True), patch.object(summary_mod, "get_image_summary_response_async", new=fake_image_response, create=True), patch.object(summary_mod, "get_text_summary_response_async", new=fake_text_response, create=True):
    cite_node = await summary_mod.summary_node_async({"t": "Cite", "c": []})
test_eq(cite_node["s"], "")

ctx.ast_json = json.dumps({"blocks": [{"t": "Cite", "c": []}]}, ensure_ascii=False)
with patch.object(summary_mod, "get_summary_response_async", new=fake_text_summary, create=True), patch.object(summary_mod, "get_image_summary_response_async", new=fake_image_response, create=True), patch.object(summary_mod, "get_text_summary_response_async", new=fake_text_response, create=True):
    await summary_mod.summary_node_main()
main_doc = json.loads(ctx.ast_json)
test_eq(main_doc["title"], "doc")
test_eq(main_doc["file_path"], str(ctx.file_path))
test_eq(main_doc["blocks"][0]["s"], "")

tmpdir.cleanup()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()